# Self-Driving Car Maze Navigation

In this notebook you will implement four classic robot navigation algorithms and test them in a Pygame simulation.

Each algorithm is implemented as a single function with the signature:

```python
def algorithm_navigate(car_x, car_y, car_angle,
                       goal_x, goal_y,
                       obstacles, car_width, car_height, dt)
    -> (new_x, new_y, new_angle)
```

| Parameter | Description |
|-----------|-------------|
| `car_x`, `car_y` | Current position of the car's center (pixels) |
| `car_angle` | Current heading in **radians** (0 = facing right, π/2 = facing up in math coords — note Pygame's y-axis is **flipped**: increasing y goes **down**) |
| `goal_x`, `goal_y` | Position of the goal (pixels) |
| `obstacles` | List of `(x, y, width, height)` rectangles |
| `car_width`, `car_height` | Dimensions of the car (pixels) |
| `dt` | Time step in seconds |

Return the **new** position and heading after one time step.

---
**Coordinate system reminder:** Pygame's y-axis increases downward.  
So to move "up" on screen you **decrease** `y`, and `math.atan2(-(dy), dx)` gives the angle in screen space.

---
After you implement all four functions, run the **Export** cell at the bottom to write them to `algorithms.py`, then launch `simulation.py` to test your work.

In [ ]:
import math
import numpy as np
from collections import deque

# ── Maze constants (must match simulation.py) ─────────────────────────────────
_MAZE_W = 720
_MAZE_H = 700

# ── Shared angle helper ───────────────────────────────────────────────────────
def _angle_diff(target, current):
    diff = target - current
    while diff >  math.pi: diff -= 2 * math.pi
    while diff < -math.pi: diff += 2 * math.pi
    return diff


# ── BFS path planner — used by Pure Pursuit and Stanley ───────────────────────
# This helper is provided for you; you do not need to modify it.
def _bfs_plan(car_x, car_y, goal_x, goal_y, obstacles, obstacle_margin):
    """
    4-directional BFS grid planner with line-of-sight path smoothing.

    4-directional (no diagonals) is critical: diagonal BFS segments can clip
    obstacle margin zones even when both cell centres are clear.

    When the car sits inside an obstacle margin zone the planner searches
    outward for the nearest free cell and starts the path from there.
    """
    CELL = 20

    cols = _MAZE_W // CELL + 2
    rows = _MAZE_H // CELL + 2

    def w2g(x, y):
        return (max(0, min(cols - 1, int(x // CELL))),
                max(0, min(rows - 1, int(y // CELL))))

    def g2w(gx, gy):
        return gx * CELL + CELL / 2, gy * CELL + CELL / 2

    def blocked(gx, gy):
        wx, wy = g2w(gx, gy)
        for ox, oy, ow, oh in obstacles:
            if (ox - obstacle_margin <= wx <= ox + ow + obstacle_margin and
                    oy - obstacle_margin <= wy <= oy + oh + obstacle_margin):
                return True
        return False

    sg = w2g(car_x, car_y)
    gg = w2g(goal_x, goal_y)

    # If start cell is inside a margin zone find the nearest free cell
    if blocked(*sg):
        found = False
        for r in range(1, 8):
            for dgx in range(-r, r + 1):
                for dgy in range(-r, r + 1):
                    if abs(dgx) == r or abs(dgy) == r:
                        cand = (sg[0] + dgx, sg[1] + dgy)
                        if (0 <= cand[0] < cols and 0 <= cand[1] < rows
                                and not blocked(*cand)):
                            sg = cand
                            found = True
                            break
                if found:
                    break
            if found:
                break

    # 4-directional BFS — axis-aligned moves never clip obstacle margins
    queue     = deque([sg])
    came_from = {sg: None}
    while queue:
        curr = queue.popleft()
        if curr == gg:
            break
        cx, cy = curr
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nxt = (cx + dx, cy + dy)
            if (0 <= nxt[0] < cols and 0 <= nxt[1] < rows
                    and nxt not in came_from and not blocked(*nxt)):
                came_from[nxt] = curr
                queue.append(nxt)

    if gg not in came_from:
        return [g2w(*sg), (goal_x, goal_y)]

    cells = []
    node  = gg
    while node is not None:
        cells.append(node)
        node = came_from[node]
    cells.reverse()

    # Start path from the nearest-unblocked grid cell centre so the car
    # steers OUT of any obstacle margin zone as its first move.
    path = [g2w(*cells[0])] + [g2w(*c) for c in cells[1:-1]] + [(goal_x, goal_y)]

    # Mild LOS smoothing — only merge segments shorter than 2 BFS cells so the
    # car stays close to the grid path and can't deviate enough to clip obstacles.
    MAX_SMOOTH = CELL * 2 + 5

    def los_clear(x1, y1, x2, y2):
        if math.hypot(x2 - x1, y2 - y1) > MAX_SMOOTH:
            return False
        steps = max(int(math.hypot(x2 - x1, y2 - y1) / 6), 2)
        for i in range(steps + 1):
            t  = i / steps
            px = x1 + t * (x2 - x1)
            py = y1 + t * (y2 - y1)
            for ox, oy, ow, oh in obstacles:
                if (ox - obstacle_margin <= px <= ox + ow + obstacle_margin and
                        oy - obstacle_margin <= py <= oy + oh + obstacle_margin):
                    return False
        return True

    smooth = [path[0]]
    i = 0
    while i < len(path) - 1:
        j = len(path) - 1
        while j > i + 1:
            if los_clear(path[i][0], path[i][1], path[j][0], path[j][1]):
                break
            j -= 1
        smooth.append(path[j])
        i = j

    return smooth


# ── Waypoint advancement helper ───────────────────────────────────────────────
# This helper is provided for you; you do not need to modify it.
def _advance_waypoints(waypoints, wp_index, car_x, car_y, car_angle, waypoint_reach):
    """
    Advance waypoint index past any waypoints that are:
    - within waypoint_reach pixels, OR
    - clearly behind the car (negative projection on heading).
    """
    fwd_x = math.cos(car_angle)
    fwd_y = -math.sin(car_angle)   # screen coords: up = negative y
    while wp_index < len(waypoints) - 1:
        wpx, wpy = waypoints[wp_index]
        dx   = wpx - car_x
        dy   = wpy - car_y
        dist = math.hypot(dx, dy)
        dot  = dx * fwd_x + dy * fwd_y
        if dist < waypoint_reach or dot < -5:
            wp_index += 1
        else:
            break
    return wp_index


# ── State reset ───────────────────────────────────────────────────────────────
def reset_algorithm_state():
    """Clear all per-algorithm state dicts (called on maze reset)."""
    for fn in [bug2_navigate, pure_pursuit_navigate,
               stanley_navigate, potential_field_navigate]:
        fn._state = None


---
## 1 · Bug2 Algorithm

### How it works

Bug2 is one of the simplest complete navigation algorithms.  It relies on two behaviours:

1. **Go-to-goal** — drive in a straight line from start **S** toward the goal **G** along the *M-line* (the straight line connecting S and G).
2. **Boundary following** — when an obstacle is detected, switch to tracing the obstacle boundary until the M-line is re-encountered at a point **closer to the goal** than where boundary following began.  Then switch back to go-to-goal.

The algorithm is provably complete (it always reaches the goal if one exists) for simple connected obstacles.

### Key variables to track
* The **M-line** — the line segment from your starting position to the goal.
* The **hit-point** — where the car first hits an obstacle.
* The **best distance** seen so far on the M-line — used to decide when to leave boundary-following mode.

### Implementation hints

1. Store state (mode, hit-point, best distance, etc.) in `bug2_navigate._state` so it persists across calls.
2. **Go-to-goal mode:**  
   - Compute the angle to the goal with `math.atan2`.  
   - Rotate toward it (clamp the turn rate) then move forward.  
   - Switch to boundary-following when an obstacle is sensed ahead.
3. **Boundary-following mode:**  
   - Keep the obstacle to one side (e.g. left wall-following: turn right when there is space, turn left when blocked).  
   - At each step check whether you are back on the M-line **and** closer to the goal than the hit-point; if so, switch back to go-to-goal.
4. Always clamp the new position inside the screen and reject positions that would put the car inside an obstacle.

In [ ]:
def bug2_navigate(car_x, car_y, car_angle, goal_x, goal_y,
                  obstacles, car_width, car_height, dt):
    """
    Bug2 algorithm.

    Returns (new_x, new_y, new_angle).
    """
    # ── Tunable constants ─────────────────────────────────────────────────────
    SPEED           = 80.0   # pixels / second
    TURN_RATE       = 2.8    # radians / second (max steering rate)
    SENSOR_DIST     = 42.0   # pixels ahead to check for obstacles
    OBSTACLE_MARGIN = 18.0   # extra clearance around obstacle rectangles

    # ── Persistent state ──────────────────────────────────────────────────────
    if getattr(bug2_navigate, '_state', None) is None:
        bug2_navigate._state = {
            'mode':               'go_to_goal',  # 'go_to_goal' or 'follow_boundary'
            'hit_point':          None,           # (x, y) where obstacle was first hit
            'best_dist_on_mline': None,           # best dist-to-goal seen on M-line
            'start_x':            car_x,
            'start_y':            car_y,
        }
    state = bug2_navigate._state

    # ── Helper functions ──────────────────────────────────────────────────────
    def point_in_obstacle(x, y):
        """Return True if (x, y) is inside any obstacle (with margin)."""
        # TODO: implement
        pass

    def obstacle_ahead(x, y, angle, dist):
        """Return True if there is an obstacle within `dist` pixels ahead.
        Sample at dist*0.5 and dist*1.0 along the heading.
        """
        # TODO: implement
        pass

    def dist_to_goal(x, y):
        return math.hypot(goal_x - x, goal_y - y)

    def on_mline(x, y):
        """Return True if (x, y) is close to the M-line AND in front of start.
        Hint: compute the perpendicular distance from (x, y) to the line through
        (state['start_x'], state['start_y']) and (goal_x, goal_y).
        Use a tolerance of ~18 px and check the dot product is positive.
        """
        # TODO: implement
        pass

    target_angle = math.atan2(-(goal_y - car_y), goal_x - car_x)

    # ── Go-to-goal mode ───────────────────────────────────────────────────────
    if state['mode'] == 'go_to_goal':
        # TODO:
        # 1. If obstacle_ahead: record hit_point + best_dist_on_mline, switch mode
        # 2. Otherwise: steer toward target_angle (clamp turn to TURN_RATE * dt)
        #    move forward, reject if point_in_obstacle
        #    If blocked: switch to follow_boundary anyway (still rotate the angle)
        # 3. Return (new_x, new_y, new_angle)
        pass

    # ── Left-hand wall follower ───────────────────────────────────────────────
    if state['mode'] == 'follow_boundary':
        # TODO:
        # left_angle = car_angle + math.pi / 2
        #
        # if obstacle_ahead:          turn right  (car_angle - TURN_RATE * dt)
        # elif no obstacle to LEFT:   turn left   (re-acquire wall around corner)
        # else:                       go straight (wall on left, clear ahead)
        #
        # Move at SPEED * 0.65 * dt in new_angle direction.
        # After moving, check on_mline: if closer to goal than best_dist_on_mline,
        #   switch back to go_to_goal.
        # If new position is blocked, still return new_angle (rotate in place).
        pass

    return car_x, car_y, car_angle


---
## 2 · Pure Pursuit

### How it works

Pure Pursuit is a **path-tracking** algorithm originally developed for autonomous vehicle steering.  It assumes a reference path is already available and steers the vehicle toward a **lookahead point** — a point on the path that lies a fixed distance *L* ahead of the current position.

The key insight is geometric: given the lookahead point, the required steering curvature is:

$$\kappa = \frac{2 \cdot d_y}{L^2}$$

where $d_y$ is the lateral offset of the lookahead point in the vehicle's local frame and $L$ is the lookahead distance.

### Steps

1. **Build a waypoint path** from start to goal that avoids obstacles.  A simple greedy planner (try heading directly toward the goal; if blocked, try angles offset by ±0.4 rad, ±0.8 rad, …) works well enough for this simulation.
2. **Find the lookahead point** — walk forward along the waypoint list until the cumulative distance exceeds `LOOKAHEAD`.
3. **Steer toward the lookahead point** — compute the bearing to it, then rotate the car at most `TURN_RATE · dt` radians toward that bearing.
4. Move forward at constant speed.

### Implementation hints

* Cache the waypoint list in `pure_pursuit_navigate._state` — recompute only when the state is cleared (maze reset).
* Advance `wp_index` whenever the car comes within `WAYPOINT_REACH` pixels of the current waypoint.
* If the new position lands inside an obstacle, clear the cached waypoints so a new path is planned next frame.

In [ ]:
def pure_pursuit_navigate(car_x, car_y, car_angle, goal_x, goal_y,
                          obstacles, car_width, car_height, dt):
    """
    Pure Pursuit controller.

    The path is planned for you by _bfs_plan (see Cell 1).
    Your task is to implement the lookahead-point selection and steering.

    Returns (new_x, new_y, new_angle).
    """
    # ── Tunable constants ─────────────────────────────────────────────────────
    SPEED           = 90.0
    LOOKAHEAD       = 28.0   # pixels — lookahead distance on path
    TURN_RATE       = 3.2    # max radians / second
    OBSTACLE_MARGIN = 20.0
    WAYPOINT_REACH  = 18.0   # waypoint is "reached" within this many pixels

    # ── Persistent state ──────────────────────────────────────────────────────
    if getattr(pure_pursuit_navigate, '_state', None) is None:
        pure_pursuit_navigate._state = {
            'waypoints':    None,
            'wp_index':     0,
            'stuck_frames': 0,
        }
    state = pure_pursuit_navigate._state

    # ── Build BFS path if needed ──────────────────────────────────────────────
    # _bfs_plan is provided in Cell 1 — do not reimplement it.
    if state['waypoints'] is None:
        state['waypoints'] = _bfs_plan(car_x, car_y, goal_x, goal_y,
                                         obstacles, OBSTACLE_MARGIN)
        state['wp_index'] = 0

    waypoints = state['waypoints']

    # ── Advance waypoint index ────────────────────────────────────────────────
    # _advance_waypoints is provided in Cell 1 — call it here.
    # TODO: call _advance_waypoints and store the result back into state['wp_index']
    wp_index = min(state['wp_index'], len(waypoints) - 1)

    # ── Find lookahead point ──────────────────────────────────────────────────
    # TODO: walk forward from wp_index searching for the first waypoint
    #       whose distance from (car_x, car_y) is >= LOOKAHEAD.
    #       Fall back to waypoints[wp_index] if no such point exists.
    lookahead_x, lookahead_y = waypoints[wp_index]   # replace with your search

    # ── Steer toward lookahead point ──────────────────────────────────────────
    # TODO:
    # 1. target_angle = math.atan2(-(lookahead_y - car_y), lookahead_x - car_x)
    # 2. angle_diff   = _angle_diff(target_angle, car_angle)
    # 3. new_angle    = car_angle + copysign(min(|angle_diff|, TURN_RATE*dt), angle_diff)
    # 4. new_x = car_x + cos(new_angle) * SPEED * dt
    #    new_y = car_y - sin(new_angle) * SPEED * dt
    # 5. Check if new position is inside any obstacle (use OBSTACLE_MARGIN).
    #    If blocked: increment stuck_frames; if > 45 clear waypoints & reset.
    #                return (car_x, car_y, new_angle)  ← rotate in place
    # 6. Otherwise: reset stuck_frames = 0; return (new_x, new_y, new_angle)
    pass

    return car_x, car_y, car_angle


---
## 3 · Stanley Controller

### How it works

The Stanley controller was used by Stanford's autonomous vehicle *Stanley* to win the 2005 DARPA Grand Challenge.  It combines two error terms:

1. **Heading error** $\psi_e$ — the difference between the car's heading and the tangent direction of the reference path.
2. **Cross-track error** $e$ — the signed perpendicular distance from the car to the nearest path segment.

The steering command is:

$$\delta = \psi_e + \arctan\!\left(\frac{k \cdot e}{v}\right)$$

where $k$ is a gain and $v$ is the vehicle speed.  At high speed the cross-track correction is small (smooth); at low speed it can be large (aggressive correction).

### Steps

1. Build the same greedy waypoint path as in Pure Pursuit.
2. Find the **closest path segment** to the car (or use the current `wp_index`).
3. Compute the **heading error**: `heading_err = _angle_diff(segment_angle, car_angle)`.
4. Compute the **cross-track error**: signed perpendicular distance from the car to the segment (positive = left of path).
5. Combine: `delta = heading_err + atan2(K * cross_track, SPEED)`.
6. Clamp `delta` to `TURN_RATE * dt` and update angle and position.

### Implementation hints

* The **cross-track error** sign convention matters — make sure positive cross-track drives the car back onto the path from the left side.
* Reuse the same greedy `build_waypoints` helper from Pure Pursuit.
* The gain `K = 2.0` and `SPEED = 85.0` are good starting values.

In [ ]:
def stanley_navigate(car_x, car_y, car_angle, goal_x, goal_y,
                     obstacles, car_width, car_height, dt):
    """
    Stanley controller.

    The path is planned for you by _bfs_plan (see Cell 1).
    Your task is to compute the heading error, cross-track error, and
    combine them into a steering command.

    Returns (new_x, new_y, new_angle).
    """
    # ── Tunable constants ─────────────────────────────────────────────────────
    SPEED           = 85.0
    K               = 2.0    # cross-track error gain
    TURN_RATE       = 3.5    # max radians / second
    OBSTACLE_MARGIN = 20.0
    WAYPOINT_REACH  = 18.0

    # ── Persistent state ──────────────────────────────────────────────────────
    if getattr(stanley_navigate, '_state', None) is None:
        stanley_navigate._state = {
            'waypoints':    None,
            'wp_index':     0,
            'stuck_frames': 0,
        }
    state = stanley_navigate._state

    # ── Build BFS path if needed ──────────────────────────────────────────────
    if state['waypoints'] is None:
        state['waypoints'] = _bfs_plan(car_x, car_y, goal_x, goal_y,
                                         obstacles, OBSTACLE_MARGIN)
        state['wp_index'] = 0

    waypoints = state['waypoints']

    # ── Advance waypoint index ────────────────────────────────────────────────
    # TODO: call _advance_waypoints and store result back into state['wp_index']
    wp_index = min(state['wp_index'], len(waypoints) - 1)

    # ── Current segment endpoints ─────────────────────────────────────────────
    tx, ty = waypoints[wp_index]
    px, py = waypoints[wp_index - 1] if wp_index > 0 else (car_x, car_y)

    # ── Compute steering errors ───────────────────────────────────────────────
    # TODO:
    # seg_dx, seg_dy = tx - px, ty - py
    # seg_len = math.hypot(seg_dx, seg_dy) + 1e-9
    # seg_angle = math.atan2(-seg_dy, seg_dx)   ← note minus sign (Pygame y)
    #
    # heading_err    = _angle_diff(seg_angle, car_angle)
    #
    # cross_track: signed perpendicular distance from car to segment.
    # Hint: ex, ey = car_x - px, car_y - py
    #       cross_track = ex * (-seg_dy / seg_len) + ey * (seg_dx / seg_len)
    #
    # cte_correction = math.atan2(K * cross_track, SPEED)
    # delta = heading_err + cte_correction   ← clamp to TURN_RATE * dt

    # ── Steer and move ────────────────────────────────────────────────────────
    # TODO:
    # new_angle = car_angle + delta
    # new_x = car_x + cos(new_angle) * SPEED * dt
    # new_y = car_y - sin(new_angle) * SPEED * dt
    #
    # If new position is inside any obstacle (use OBSTACLE_MARGIN):
    #   increment stuck_frames; if > 45 clear waypoints & reset.
    #   return (car_x, car_y, new_angle)   ← rotate in place
    # Otherwise: reset stuck_frames = 0; return (new_x, new_y, new_angle)
    pass

    return car_x, car_y, car_angle


---
## 4 · Potential Field Controller

### How it works

Potential Field navigation treats the robot's workspace as a scalar potential field:

* The **goal** creates an **attractive** potential that decreases with distance — like a gravity well pulling the robot in.
* Each **obstacle** creates a **repulsive** potential that increases as the robot gets close — like a magnetic repulsion.

The robot follows the **negative gradient** of the total potential, which gives a force vector pointing (approximately) toward the goal while pushing away from obstacles.

$$\mathbf{F}_{att} = k_{att} \cdot (\mathbf{q}_{goal} - \mathbf{q})$$

$$\mathbf{F}_{rep} = k_{rep} \left(\frac{1}{d} - \frac{1}{d_0}\right) \frac{1}{d^2} \hat{\mathbf{d}} \quad \text{if } d < d_0$$

where $d$ is the distance to the nearest obstacle surface, $d_0$ is the influence radius, and $\hat{\mathbf{d}}$ is the unit vector **away** from the obstacle.

The total force is $\mathbf{F} = \mathbf{F}_{att} + \sum \mathbf{F}_{rep}$. Normalise it to get a direction, then move at constant speed.

### Known limitation

Potential fields can get stuck in **local minima** (places where attractive and repulsive forces exactly cancel). If this happens, add a small random perturbation or implement a simple escape strategy.

### Implementation hints

* For each obstacle rectangle, the **closest point** on the rectangle to the car is `(clamp(car_x, ox, ox+ow), clamp(car_y, oy, oy+oh))`.
* Use `K_ATT = 1.0`, `K_REP = 8000.0`, `D0 = 60.0` as starting values.
* Normalise the total force vector before using it as a heading target.
* If the new position lands inside an obstacle, try a small angular perturbation.

In [ ]:
def potential_field_navigate(car_x, car_y, car_angle, goal_x, goal_y,
                             obstacles, car_width, car_height, dt):
    """
    Potential Field controller.

    Returns (new_x, new_y, new_angle).
    """
    # ── Tunable constants ─────────────────────────────────────────────────────
    SPEED           = 78.0
    TURN_RATE       = 3.2
    K_ATT           = 1.0      # attractive gain
    K_REP           = 9000.0   # repulsive gain
    D0              = 65.0     # repulsion influence radius (pixels)
    OBSTACLE_MARGIN = 15.0
    ESCAPE_THRESH   = 18       # stuck frames before escape kick

    # ── Persistent state ──────────────────────────────────────────────────────
    if getattr(potential_field_navigate, '_state', None) is None:
        potential_field_navigate._state = {'stuck_frames': 0}
    state = potential_field_navigate._state

    # ── Attractive force ──────────────────────────────────────────────────────
    # TODO: att_x = K_ATT * (goal_x - car_x)
    #       att_y = K_ATT * (goal_y - car_y)
    att_x, att_y = 0.0, 0.0

    # ── Repulsive forces + nearest-obstacle tracking ──────────────────────────
    rep_x, rep_y  = 0.0, 0.0
    nearest_dist  = float('inf')
    nearest_esc_x = 1.0   # unit vector pointing away from nearest obstacle
    nearest_esc_y = 0.0

    for obs in obstacles:
        ox, oy, ow, oh = obs
        # TODO:
        # 1. closest point on rectangle to car:
        #      closest_x = max(ox, min(car_x, ox + ow))
        #      closest_y = max(oy, min(car_y, oy + oh))
        # 2. dx, dy = car_x - closest_x, car_y - closest_y
        #    dist = math.hypot(dx, dy) + 1e-9
        # 3. If dist < D0:
        #      mag    = K_REP * (1/dist - 1/D0) / dist**2
        #      rep_x += mag * dx / dist
        #      rep_y += mag * dy / dist
        # 4. Track nearest obstacle for escape direction:
        #      if dist < nearest_dist: update nearest_dist, nearest_esc_x/y
        pass

    # ── Escape mode: bypass margin check and push away from nearest obstacle ──
    # When the car is trapped in a local minimum, this kicks in after
    # ESCAPE_THRESH stuck frames and forces movement directly away from the
    # nearest obstacle at 3× speed, ignoring the margin check.
    if state['stuck_frames'] >= ESCAPE_THRESH:
        # TODO:
        # escape_angle = math.atan2(-nearest_esc_y, nearest_esc_x)
        # new_x = car_x + cos(escape_angle) * SPEED * 3.0 * dt
        # new_y = car_y - sin(escape_angle) * SPEED * 3.0 * dt
        # Reset stuck_frames to 0 only if new position is NOT blocked.
        # Return (new_x, new_y, escape_angle) regardless.
        pass

    # ── Normal potential-field move ───────────────────────────────────────────
    # TODO:
    # force_x, force_y = att_x + rep_x, att_y + rep_y
    # Normalise to unit vector (add 1e-9 to magnitude to avoid div-by-zero).
    # target_angle = math.atan2(-force_y, force_x)
    # angle_diff   = _angle_diff(target_angle, car_angle)
    # new_angle    = car_angle + copysign(min(|angle_diff|, TURN_RATE*dt), angle_diff)
    # new_x = car_x + cos(new_angle) * SPEED * dt
    # new_y = car_y - sin(new_angle) * SPEED * dt
    #
    # If blocked OR movement < 0.05 px: increment stuck_frames; return old pos.
    # Otherwise: decrement stuck_frames toward 0; return (new_x, new_y, new_angle).
    pass

    return car_x, car_y, car_angle


---
## Export to `algorithms.py`

Run the cell below once you have implemented all four functions.  It extracts the function source code and writes `algorithms.py` in the same directory, which `simulation.py` imports.

Then open a terminal and run:

```bash
python simulation.py
```

Use the dropdown in the sidebar to select an algorithm, and the **Reset Maze** button to randomise obstacles.

In [ ]:
import inspect

# Export all functions — including the BFS helpers that the path-following
# algorithms depend on.  Students only need to implement the four navigate
# functions; the helpers and the _MAZE_W / _MAZE_H constants are written
# automatically by this cell.
functions = [
    bug2_navigate,
    _bfs_plan,
    _advance_waypoints,
    pure_pursuit_navigate,
    stanley_navigate,
    potential_field_navigate,
    _angle_diff,
    reset_algorithm_state,
]

header = """import math
import numpy as np
from collections import deque

# Must match simulation.py
_MAZE_W = 720
_MAZE_H = 700


"""

output_path = 'algorithms.py'

with open(output_path, 'w') as f:
    f.write(header)
    for fn in functions:
        src = inspect.getsource(fn)
        f.write(src)
        f.write('\n\n')

print(f'Written to {output_path}')
print('Now run:  python simulation.py')
